In [1]:
#============================================
# Celda 1 — Imports y carga del JSON
#============================================
import json, glob, pandas as pd, numpy as np, os
from pathlib import Path
from datetime import datetime

ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "requirements.txt").exists():
    raise FileNotFoundError(f"No se encontró requirements.txt subiendo desde {Path.cwd()}")
os.chdir(ROOT)
print(f"✅ ROOT: {ROOT}")


def ultimo_json(carpeta_base: str) -> str:
    archivos = glob.glob(f'{carpeta_base}/**/*.json', recursive=True)
    assert len(archivos) > 0, f"❌ No hay JSONs en {carpeta_base}"
    def parse_fecha(ruta):
        try:
            return datetime.strptime(Path(ruta).stem, "%d_%m_%Y")
        except ValueError:
            return datetime.min
    return max(archivos, key=parse_fecha)


f = ultimo_json('data/ambiental')
data = json.load(open(f))
print(f'Archivo cargado: {f}')
print(f'Keys disponibles: {list(data.keys())}')

assert 'observacion_meteorologica' in data, \
    f"❌ Key 'observacion_meteorologica' no encontrada. Keys: {list(data.keys())}"

_meteo = data['observacion_meteorologica']
AEMET_OK = 'estaciones' in _meteo and len(_meteo['estaciones']) > 0
estaciones = _meteo.get('estaciones', [])

if not AEMET_OK:
    print(f"⚠️ AEMET sin estaciones en esta captura — análisis meteorológico saltado")
    print(f"   Estado AEMET: {_meteo.get('estado', _meteo.get('error', 'desconocido'))}")
else:
    print(f"✅ AEMET OK — {len(estaciones)} estaciones")

assert 'calidad_aire' in data, \
    f"❌ Key 'calidad_aire' no encontrada. Keys: {list(data.keys())}"

# ── Variables globales siempre disponibles para el resto del notebook ──
ciudades   = data['calidad_aire'].get('ciudades', [])   # ← NUEVO
WAQI_OK    = len(ciudades) > 0                          # ← NUEVO

print(f"✅ Estructura JSON validada")
if AEMET_OK: print(f"   Estaciones AEMET: {len(estaciones)}")
print(f"   Ciudades WAQI:    {len(ciudades)}")

✅ ROOT: /workspaces/TFG_Spain-s_Migratory_Flow
Archivo cargado: data/ambiental/2026/06/20_06_2026.json
Keys disponibles: ['timestamp', 'observacion_meteorologica', 'calidad_aire']
⚠️ AEMET sin estaciones en esta captura — análisis meteorológico saltado
   Estado AEMET: 429
✅ Estructura JSON validada
   Ciudades WAQI:    30


In [2]:
#============================================
# Celda 2 — Explorar estructura AEMET
#============================================
aemet_raw = data['observacion_meteorologica']

if not AEMET_OK:
    print("⚠️  AEMET no disponible en esta captura — celda saltada")
else:
    estaciones = aemet_raw['estaciones']
    print(f'Total estaciones: {aemet_raw["total_estaciones"]}')
    print(f'Timestamp captura: {aemet_raw["timestamp_captura"]}')
    print(f'\nKeys de una estación:')
    print(list(estaciones[0].keys()))
    print(f'\nEjemplo estación[0]:')
    print(json.dumps(estaciones[0], indent=2, ensure_ascii=False))

⚠️  AEMET no disponible en esta captura — celda saltada


In [3]:
#============================================
# Celda 3 — Parsear AEMET a DataFrame
#============================================
if not AEMET_OK:
    print("⚠️  AEMET no disponible — celda saltada")
    df_aemet = pd.DataFrame()
else:
    df_aemet = pd.DataFrame(estaciones)

    # Convertir timestamp
    df_aemet['fint'] = pd.to_datetime(df_aemet['fint'], utc=True, errors='coerce')
    df_aemet['fecha'] = df_aemet['fint'].dt.date

    # Columnas numéricas
    cols_num = ['lat','lon','alt','ta','tamin','tamax','prec','hr',
                'vv','vmax','dv','dmax','pres','pres_nmar','vis','inso','ts','tpr']
    for c in cols_num:
        if c in df_aemet.columns:
            df_aemet[c] = pd.to_numeric(df_aemet[c], errors='coerce')

    print(f'Shape: {df_aemet.shape}')
    print(f'Columnas: {list(df_aemet.columns)}')
    display(df_aemet.head())

⚠️  AEMET no disponible — celda saltada


In [4]:
#============================================
# Celda 4 — Estadísticas básicas AEMET
#============================================
if not AEMET_OK or df_aemet.empty:
    print("⚠️  AEMET no disponible — celda saltada")
else:
    print(f'Rango temperatura:  {df_aemet["ta"].min():.1f}°C — {df_aemet["ta"].max():.1f}°C')
    print(f'Media temperatura:  {df_aemet["ta"].mean():.1f}°C')
    print(f'\nEstaciones con precipitación > 0: {(df_aemet["prec"] > 0).sum()}')

⚠️  AEMET no disponible — celda saltada


In [5]:
#============================================
# Celda 5 — Resumen del dataset ambiental
#============================================

# ── Guardia: reconstruir df_waqi si no existe en el kernel ──
if 'df_waqi' not in dir() or df_waqi is None:
    print("⚠️  df_waqi no encontrado en memoria — reconstruyendo desde data...")
    ciudades_raw = data['calidad_aire'].get('ciudades', [])
    if ciudades_raw:
        df_waqi = pd.DataFrame(ciudades_raw)
        if 'timestamp' in df_waqi.columns:
            df_waqi['timestamp'] = pd.to_datetime(df_waqi['timestamp'], errors='coerce')
        print(f"   ✅ df_waqi reconstruido: {df_waqi.shape}")
    else:
        df_waqi = pd.DataFrame()
        print("   ❌ Sin datos WAQI en el JSON — df_waqi vacío")

WAQI_OK = not df_waqi.empty

# ── Resumen ──
if not AEMET_OK and not WAQI_OK:
    print("❌ Ni AEMET ni WAQI disponibles — sin datos para resumir")
    resumen = pd.DataFrame()

elif not AEMET_OK or df_aemet.empty if 'df_aemet' in dir() else True:
    print("⚠️  AEMET no disponible — resumen parcial (solo WAQI)")
    resumen = pd.DataFrame([{
        'fuente':    'waqi_ciudades',
        'filas':     len(df_waqi),
        'columnas':  len(df_waqi.columns),
        'periodos':  df_waqi['timestamp'].nunique() if 'timestamp' in df_waqi.columns else 0,
        'nulos_%':   round(df_waqi.isnull().sum().sum() / (df_waqi.shape[0] * df_waqi.shape[1]) * 100, 1)
    }])
else:
    resumen = pd.DataFrame([
        {
            'fuente':    'aemet_estaciones',
            'filas':     len(df_aemet),
            'columnas':  len(df_aemet.columns),
            'periodos':  df_aemet['fecha'].nunique() if 'fecha' in df_aemet.columns else 0,
            'nulos_%':   round(df_aemet.isnull().sum().sum() / (df_aemet.shape[0] * df_aemet.shape[1]) * 100, 1)
        },
        {
            'fuente':    'waqi_ciudades',
            'filas':     len(df_waqi),
            'columnas':  len(df_waqi.columns),
            'periodos':  df_waqi['timestamp'].nunique() if 'timestamp' in df_waqi.columns else 0,
            'nulos_%':   round(df_waqi.isnull().sum().sum() / (df_waqi.shape[0] * df_waqi.shape[1]) * 100, 1)
        },
    ])

print(resumen.to_string(index=False))

⚠️  df_waqi no encontrado en memoria — reconstruyendo desde data...
   ✅ df_waqi reconstruido: (30, 17)
⚠️  AEMET no disponible — resumen parcial (solo WAQI)
       fuente  filas  columnas  periodos  nulos_%
waqi_ciudades     30        17         9      3.9


In [6]:
#============================================
# Celda 6 — Parsear WAQI a DataFrame
#============================================

# ── Guardia: asegurar que 'ciudades' existe ──
if 'ciudades' not in dir() or not ciudades:
    print("⚠️  'ciudades' no en memoria — extrayendo desde data...")
    ciudades = data['calidad_aire'].get('ciudades', [])
    if not ciudades:
        raise ValueError("❌ No hay datos WAQI en el JSON. Revisa que Celda 1 cargó correctamente.")
    print(f"   ✅ {len(ciudades)} ciudades recuperadas")

df_waqi = pd.DataFrame(ciudades)

# Convertir tipos
df_waqi['timestamp'] = pd.to_datetime(df_waqi['timestamp'], errors='coerce')

cols_num = ['lat', 'lon', 'aqi', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO',
            'temperatura', 'humedad', 'presion', 'viento']
for c in cols_num:
    if c in df_waqi.columns:
        df_waqi[c] = pd.to_numeric(df_waqi[c], errors='coerce')

print(f'Shape: {df_waqi.shape}')
print(f'Columnas: {list(df_waqi.columns)}')
df_waqi.head(10)

Shape: (30, 17)
Columnas: ['ciudad', 'nombre', 'lat', 'lon', 'timestamp', 'aqi', 'dominante', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO', 'temperatura', 'humedad', 'presion', 'viento']


,ciudad,nombre,lat,lon,timestamp,aqi,dominante,NO2,PM25,PM10,O3,SO2,CO,temperatura,humedad,presion,viento
0,madrid,Madrid,40.416775,-3.703790,2026-06-20 08:00:00,53.0,pm25,5.1,53.0,30.0,26.4,1.6,0.1,30.0,33.6,1019.1,2.1
1,barcelona,"Barcelona (Eixample), Catalunya, Spain",41.385343,2.153822,2026-06-20 12:00:00,31.0,o3,8.7,34.0,19.0,30.5,1.1,0.1,30.5,39.5,1023.5,1.0
2,valencia,"Pista de Silla, València, Valencia, Spain",39.456111,-0.375833,2026-06-20 07:00:00,57.0,pm25,3.7,57.0,28.0,19.9,0.6,NaN,26.6,71.0,NaN,0.1
3,sevilla,"Ranilla, Sevilla, Spain",37.384250,-5.959620,2026-06-20 12:00:00,66.0,pm25,1.0,66.0,39.0,29.7,1.6,0.1,33.0,46.0,1013.3,2.5
4,bilbao,"Mazarredo, Bilbao, País Vasco, Spain",43.267500,-2.935200,2026-06-20 11:00:00,0.0,co,3.7,65.0,23.0,NaN,2.6,0.1,28.0,61.0,1019.0,1.0
5,zaragoza,"El Picarral, Zaragoza, Spain",41.670278,-0.871111,2026-06-20 11:00:00,39.0,pm25,7.2,39.0,NaN,35.0,NaN,1.2,31.1,28.0,1020.4,3.3
6,malaga,"Carranque, Malaga, Spain",36.719640,-4.447500,2026-06-20 12:00:00,38.0,pm25,6.4,38.0,29.0,30.0,2.1,0.1,29.4,57.0,1016.6,2.0
7,murcia,"San Basilio Murcia Ciudad, Murcia, Spain",37.993960,-1.144628,2026-06-20 12:00:00,59.0,pm25,5.3,59.0,40.0,38.3,1.8,0.1,31.6,49.0,1017.4,7.5
8,palma,"foners, Baleares, Spain",39.571250,2.657028,2026-06-20 07:00:00,46.0,pm25,3.2,46.0,18.0,37.0,1.4,0.1,34.5,25.0,1019.0,5.6
9,alicante,"Rabassa, Alacant, Valencia, Spain",38.351111,-0.513889,2026-06-20 08:00:00,24.0,o3,3.7,17.0,20.0,24.4,0.6,0.1,31.0,53.5,1017.0,5.1


In [7]:
#============================================
# Celda 7 — Análisis rápido WAQI
#============================================
def nivel_aqi(aqi):
    if pd.isna(aqi):       return 'Sin datos'
    elif aqi <= 50:        return '🟢 Bueno'
    elif aqi <= 100:       return '🟡 Moderado'
    elif aqi <= 150:       return '🟠 Insalubre sensibles'
    elif aqi <= 200:       return '🔴 Insalubre'
    else:                  return '🟣 Muy insalubre'

df_waqi['nivel_aqi'] = df_waqi['aqi'].apply(nivel_aqi)

print('Ciudades por nivel AQI:')
print(df_waqi['nivel_aqi'].value_counts().to_string())

print(f'\nTop 5 peor AQI:')
print(df_waqi.nlargest(5, 'aqi')[['nombre','aqi','dominante','PM25','PM10','O3']].to_string(index=False))

print(f'\nTop 5 mejor AQI:')
print(df_waqi.nsmallest(5, 'aqi')[['nombre','aqi','dominante','PM25','PM10','O3']].to_string(index=False))

print(f'\nValores nulos por columna:')
print(df_waqi.isnull().sum()[df_waqi.isnull().sum() > 0])


Ciudades por nivel AQI:
nivel_aqi
🟢 Bueno       22
🟡 Moderado     7
Sin datos      1

Top 5 peor AQI:
                                   nombre  aqi dominante  PM25  PM10   O3
                  Ranilla, Sevilla, Spain 66.0      pm25  66.0  39.0 29.7
                  La Orden, Huelva, Spain 62.0      pm25  62.0  35.0 14.7
 San Basilio Murcia Ciudad, Murcia, Spain 59.0      pm25  59.0  40.0 38.3
Pista de Silla, València, Valencia, Spain 57.0      pm25  57.0  28.0 19.9
                                   Madrid 53.0      pm25  53.0  30.0 26.4

Top 5 mejor AQI:
                                   nombre  aqi dominante  PM25  PM10    O3
     Mazarredo, Bilbao, País Vasco, Spain  0.0        co  65.0  23.0   NaN
Vila Velha - IBES, Espírito Santo, Brazil  5.0        o3   NaN   NaN  4.83
       Lope de Vega, Vigo, Galicia, Spain 13.0      pm25  13.0  12.0 10.10
  Tarragona (Bonavista), Catalunya, Spain 17.0      pm10  30.0  17.0 12.00
               Tome Cano, Canarias, Spain 20.0        o3  17.

In [8]:
#============================================
# Celda 8 — Resumen calidad de datos
#============================================

filas_resumen = []

# ── Bloque AEMET (solo si df_aemet existe y tiene la columna 'fint') ──
if 'df_aemet' in dir() and not df_aemet.empty and 'fint' in df_aemet.columns:
    col_periodo_aemet = 'fint'
elif 'df_aemet' in dir() and not df_aemet.empty and 'fecha' in df_aemet.columns:
    col_periodo_aemet = 'fecha'   # fallback: columna 'fecha' derivada
else:
    col_periodo_aemet = None

if col_periodo_aemet:
    filas_resumen.append({
        'tabla':     'aemet_estaciones',
        'filas':     len(df_aemet),
        'columnas':  len(df_aemet.columns),
        'periodos':  df_aemet[col_periodo_aemet].nunique(),
        'nulos_%':   round(df_aemet.isnull().sum().sum() / (df_aemet.shape[0] * df_aemet.shape[1]) * 100, 1)
    })
else:
    print("⚠️  df_aemet no disponible o sin columna de periodo — fila AEMET omitida")

# ── Bloque WAQI ──
if 'df_waqi' in dir() and not df_waqi.empty:
    filas_resumen.append({
        'tabla':     'waqi_ciudades',
        'filas':     len(df_waqi),
        'columnas':  len(df_waqi.columns),
        'periodos':  df_waqi['timestamp'].nunique() if 'timestamp' in df_waqi.columns else 0,
        'nulos_%':   round(df_waqi.isnull().sum().sum() / (df_waqi.shape[0] * df_waqi.shape[1]) * 100, 1)
    })
else:
    print("⚠️  df_waqi no disponible — fila WAQI omitida")

resumen = pd.DataFrame(filas_resumen)
print(resumen.to_string(index=False))

⚠️  df_aemet no disponible o sin columna de periodo — fila AEMET omitida
        tabla  filas  columnas  periodos  nulos_%
waqi_ciudades     30        18         9      3.9


In [9]:
#============================================
# Celda 9 — Guardar parquets
#============================================
os.makedirs('output/ambiental/01-raw', exist_ok=True)   # ← sin ../../

df_aemet.to_parquet('output/ambiental/01-raw/raw_aemet_estaciones.parquet', index=False)
print(f'✅ Guardado: output/ambiental/01-raw/raw_aemet_estaciones.parquet ({len(df_aemet)} filas)')

df_waqi.to_parquet('output/ambiental/raw_waqi_ciudades.parquet', index=False)
print(f'✅ Guardado: output/ambiental/raw_waqi_ciudades.parquet ({len(df_waqi)} filas)')

print('\n🎉 Todos los parquets ambiental guardados en output/')

✅ Guardado: output/ambiental/01-raw/raw_aemet_estaciones.parquet (0 filas)
✅ Guardado: output/ambiental/raw_waqi_ciudades.parquet (30 filas)

🎉 Todos los parquets ambiental guardados en output/


In [10]:
#============================================
# Celda 10 — Verificar lectura de parquets
#============================================
import pandas as pd
from pathlib import Path

parquets = {
    'aemet': 'output/ambiental/01-raw/raw_aemet_estaciones.parquet',
    'waqi':  'output/ambiental/raw_waqi_ciudades.parquet',
}

for nombre, ruta in parquets.items():
    p = Path(ruta)
    if not p.exists():
        print(f"❌ No encontrado: {ruta}")
        continue
    df = pd.read_parquet(ruta)
    print(f"✅ {nombre:6s} → {df.shape[0]:>6} filas × {df.shape[1]:>2} cols | "
          f"nulos: {round(df.isnull().sum().sum() / max(df.size,1)*100,1)}% | "
          f"columnas: {list(df.columns)}")

print("\n🎯 Parquets listos para el notebook 02_")

✅ aemet  →      0 filas ×  0 cols | nulos: 0.0% | columnas: []
✅ waqi   →     30 filas × 18 cols | nulos: 3.9% | columnas: ['ciudad', 'nombre', 'lat', 'lon', 'timestamp', 'aqi', 'dominante', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO', 'temperatura', 'humedad', 'presion', 'viento', 'nivel_aqi']

🎯 Parquets listos para el notebook 02_
